Testing some unsupervised model and supervised model for predicting the result of F1 race based on historical data from prevous seasons from 2022 to 2025, right after changing the regulations
in 2022 season.
With f1 api : FastF1

In [25]:
#@title Przydatne moduły do zaimportowania

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

# Nice printing
import pprint

# Data processing
import numpy as np
import pandas as pd

# Data transformation
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from numpy import log

# Data preparation for learning, model selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV
import sklearn.model_selection as ms
# Linear regression
from sklearn import linear_model
from sklearn.linear_model import Ridge, Lasso

# Processing pipelining
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline

# Quality indicators
from sklearn import metrics

Getting the data and preprocessing it:

Data on which I want to make trainig for the model:

- Qualifying

- Last year qualifying

- driver standings

Data on which I want to make predictions:

- Qualifying results

- Last year race results in this circuit

- Driver standings

- Constructor standings

In [26]:
import fastf1
import fastf1.plotting

fastf1.plotting.setup_mpl(mpl_timedelta_support=True,color_scheme='fastf1')

In [74]:
from fastf1.ergast import Ergast
ergast=Ergast()
year=2024
driver_standings=ergast.get_driver_standings(season=year, round=23)

driver_standings_df=driver_standings.content[0]
drop_list=['driverId','driverUrl','givenName','familyName','dateOfBirth','driverNationality','constructorIds','constructorUrls','constructorNationalities', 'positionText']
driver_standings_df.drop(drop_list,axis=1, inplace=True)
driver_standings_df

,position,points,wins,driverNumber,driverCode,constructorNames
0,1,429.0,9,33,VER,[Red Bull]
1,2,349.0,3,4,NOR,[McLaren]
2,3,341.0,3,16,LEC,[Ferrari]
3,4,291.0,2,81,PIA,[McLaren]
4,5,272.0,2,55,SAI,[Ferrari]
5,6,235.0,2,63,RUS,[Mercedes]
6,7,211.0,2,44,HAM,[Mercedes]
7,8,152.0,0,11,PER,[Red Bull]
8,9,68.0,0,14,ALO,[Aston Martin]
9,10,37.0,0,27,HUL,[Haas F1 Team]


In [75]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder

unknown_code=-1

driver_encoder=OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=unknown_code)
constructor_encoder=LabelEncoder()
# one_hot_encoding=OneHotEncoder(handle_unknown='ignore', sparse_output=False)
driver_standings_df['constructorNames']=driver_standings_df['constructorNames'].str.get(0)
driver_standings_df['constructorNames']=constructor_encoder.fit_transform(driver_standings_df[['constructorNames']])
driver_standings_df['driverCode']=driver_encoder.fit_transform(driver_standings_df[['driverCode']])
# driver_standings_df['constructorNames']=one_hot_encoding.fit_transform(driver_standings_df[['constructorNames']])

driver_standings_df


c:\Users\mateu\Desktop\python\ML_practice\1\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,position,points,wins,driverNumber,driverCode,constructorNames
0,1,429.0,9,33,21.0,7
1,2,349.0,3,4,11.0,4
2,3,341.0,3,16,9.0,2
3,4,291.0,2,81,14.0,4
4,5,272.0,2,55,17.0,2
5,6,235.0,2,63,16.0,5
6,7,211.0,2,44,6.0,5
7,8,152.0,0,11,13.0,7
8,9,68.0,0,14,1.0,1
9,10,37.0,0,27,7.0,3


In [76]:
year=2024

quali=fastf1.get_session(year,'AbuDhabi','Q')
quali.load()
# race_df=race.content[0]
# race_df
quali

core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '55', '27', '1', '10', '63', '14', '77', '11', '22', '30', '18', '16', '20', '23', '24', '44', '43', '61']


2024 Season Round 24: Abu Dhabi Grand Prix - Qualifying

In [80]:
from fastf1.core import Laps
fastest_laps_list=list()
drivers=pd.unique(quali.laps['Driver'])
drivers
for drv in drivers:
    drvs_fastest_lap=quali.laps.pick_drivers(drv).pick_fastest()
    fastest_laps_list.append(drvs_fastest_lap)
fastest_laps=Laps(fastest_laps_list).sort_values(by="LapTime").reset_index(drop=True)

pole_lap=fastest_laps.pick_fastest()
fastest_laps['LapTimeDelta']=fastest_laps['LapTime']-pole_lap['LapTime']
# print(fastest_laps[['Driver', 'LapTime', 'LapTimeDelta']])
quali_data=pd.DataFrame(data=fastest_laps[['Driver', 'LapTime']])
# quali_data.drop(index=19,inplace=True)
quali_data.rename(columns={'Driver':'driverCode'},inplace=True)
encoded_driver=driver_encoder.transform(quali_data[['driverCode']]).flatten().astype(int)
quali_data['driverCode']=encoded_driver
quali_data


,driverCode,LapTime
0,11,0 days 00:01:22.595000
1,14,0 days 00:01:22.804000
2,17,0 days 00:01:22.824000
3,7,0 days 00:01:22.886000
4,21,0 days 00:01:22.945000
5,5,0 days 00:01:22.984000
6,16,0 days 00:01:23.132000
7,1,0 days 00:01:23.196000
8,3,0 days 00:01:23.204000
9,13,0 days 00:01:23.264000


In [81]:
race=fastf1.get_session(year, 'AbuDhabi','R')
race.load()

race_results=race.results
race_results['drivercode']=race_results['Abbreviation']
y=race_results[['Position']]
y


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']


,Position
4,1.0
55,2.0
16,3.0
44,4.0
63,5.0
1,6.0
10,7.0
27,8.0
14,9.0
81,10.0


In [85]:
all_data=pd.merge(quali_data,driver_standings_df, on='driverCode', how='left')
all_data.dropna()
all_data.drop('LapTime', axis=1, inplace=True)
X=all_data
all_data

,driverCode,position,points,wins,driverNumber,constructorNames
0,11,2.0,349.0,3.0,4.0,4.0
1,14,4.0,291.0,2.0,81.0,4.0
2,17,5.0,272.0,2.0,55.0,2.0
3,7,10.0,37.0,0.0,27.0,3.0
4,21,1.0,429.0,9.0,33.0,7.0
5,5,11.0,36.0,0.0,10.0,0.0
6,16,6.0,235.0,2.0,63.0,5.0
7,1,9.0,68.0,0.0,14.0,1.0
8,3,22.0,0.0,0.0,77.0,8.0
9,13,8.0,152.0,0.0,11.0,7.0


In [97]:
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

,driverCode,position,points,wins,driverNumber,constructorNames
0,11,2.0,349.0,3.0,4.0,4.0
17,6,7.0,211.0,2.0,44.0,5.0
15,0,16.0,12.0,0.0,23.0,9.0
1,14,4.0,291.0,2.0,81.0,4.0


In [87]:
from sklearn.ensemble import RandomForestRegressor

rf_model=RandomForestRegressor(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)

c:\Users\mateu\Desktop\python\ML_practice\1\.venv\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [89]:
y_pred=rf_model.predict(X_test)

In [90]:
print(f'MSE: {metrics.mean_squared_error(y_test,y_pred): .2f}')
print(f' r^2: {metrics.r2_score(y_test,y_pred):.2f}')

MSE:  57.34
 r^2: 0.06


In [91]:
y_pred_rounded=np.round(y_pred).astype(int)

In [98]:
race_results=X_test[['driverCode']].copy()
race_results.reset_index(drop=True,inplace=True)
race_results['Predicted_positions']=y_pred_rounded
race_results

,driverCode,Predicted_positions
0,11,9
1,6,6
2,0,16
3,14,6


In [93]:
print(race_results.head())

   driverCode  Predicted_positions
0          11                    9
1           6                    6
2           0                   16
3          14                    6


In [134]:


encoded_position=race_results['driverCode'].to_numpy().reshape(-1,1)
# encoded_position
decoded=driver_encoder.inverse_transform(encoded_position).ravel()
decoded
race_results['driverCode']=decoded
final_predictions=race_results[['driverCode','Predicted_positions']].sort_values(by='Predicted_positions')
print(final_predictions)

  driverCode  Predicted_positions
1        HAM                    6
3        PIA                    6
0        NOR                    9
2        ALB                   16
